# Baseline Models - TF-IDF + Classical ML

Before fine-tuning PubMedBERT, let's see what simple models can do. TF-IDF features with LogReg and SVM.

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

In [2]:
ds = load_dataset('pubmed_rct', '200k')

train_texts = ds['train']['sentence']
train_labels = ds['train']['label']
test_texts = ds['test']['sentence']
test_labels = ds['test']['label']

print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

Train: 180040, Test: 30135


In [3]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)

print(f'TF-IDF shape: {X_train.shape}')

TF-IDF shape: (180040, 50000)


## Logistic Regression

In [4]:
lr = LogisticRegression(C=1.0, max_iter=1000, solver='saga', n_jobs=-1)
lr.fit(X_train, train_labels)
lr_preds = lr.predict(X_test)

print(f'Logistic Regression accuracy: {accuracy_score(test_labels, lr_preds):.4f}')
print()
print(classification_report(test_labels, lr_preds, digits=4))

Logistic Regression accuracy: 0.8217

                precision    recall  f1-score   support

    BACKGROUND     0.7842    0.7691    0.7766      4378
     OBJECTIVE     0.8104    0.7423    0.7749      3524
       METHODS     0.8291    0.8847    0.8560      8892
       RESULTS     0.8434    0.8512    0.8473      9741
   CONCLUSIONS     0.8013    0.7249    0.7612      3600

      accuracy                         0.8217     30135
     macro avg     0.8137    0.7944    0.8032     30135
  weighted avg     0.8212    0.8217    0.8208     30135


82% accuracy. METHODS and RESULTS do best (makes sense, they have distinct vocabulary). CONCLUSIONS is hardest to separate from BACKGROUND/OBJECTIVE.

## Linear SVM

In [5]:
svm = LinearSVC(C=1.0, max_iter=2000)
svm.fit(X_train, train_labels)
svm_preds = svm.predict(X_test)

print(f'SVM accuracy: {accuracy_score(test_labels, svm_preds):.4f}')
print()
print(classification_report(test_labels, svm_preds, digits=4))

SVM accuracy: 0.8284

                precision    recall  f1-score   support

    BACKGROUND     0.7931    0.7748    0.7838      4378
     OBJECTIVE     0.8193    0.7512    0.7838      3524
       METHODS     0.8342    0.8923    0.8623      8892
       RESULTS     0.8501    0.8574    0.8537      9741
   CONCLUSIONS     0.8078    0.7341    0.7692      3600

      accuracy                         0.8284     30135
     macro avg     0.8209    0.8020    0.8106     30135
  weighted avg     0.8279    0.8284    0.8274     30135


SVM edges out LogReg slightly at 82.8%. Both struggle most with CONCLUSIONS (F1 ~0.77).

The METHODS class gets the highest F1 in both models, probably because method-specific terms like "randomized", "placebo-controlled", "enrolled" are very distinctive.

Baseline to beat: **82.8% accuracy, 0.81 macro F1**

Let's see if PubMedBERT can do meaningfully better, especially on the harder classes.
# SVM confirmed slightly better